# Run Object Detection Project

## 1. Settings

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
BRANCH = ""  # Optional, for example: "main"
PROJECT_DIR = Path("/kaggle/working/object-detection")

# Optional. Leave as None to auto-detect under /kaggle/input.
# Examples:
# DATASET_DIR = "/kaggle/input/object-detection-public/public"
# DATASET_DIR = "/kaggle/input/object-detection-public"
DATASET_DIR = None

IMAGE_SIZE = 512  # Anchor-free model; no anchor scaling is needed.
EPOCHS = 60
BATCH_SIZE = 16
NUM_WORKERS = 2
LR = 1e-3
WEIGHT_DECAY = 1e-4
USE_AMP = True
USE_PRETRAINED_BACKBONE = False
FREEZE_BACKBONE_STEM = False

# Set these to small values for a quick smoke test. Use 0 for full train/validation.
MAX_TRAIN_BATCHES = 0
MAX_VAL_BATCHES = 0

CONF_THRESHOLD = 0.05
NMS_THRESHOLD = 0.50

# Optional hidden/test image directory. Leave as None if you only want val predictions.
TEST_IMAGE_DIR = None
FINAL_PREDICTIONS = Path("/kaggle/working/predictions.json")


## 2. Clone Code And Install Requirements

In [ ]:
import os
import subprocess
import sys

if not PROJECT_DIR.exists():
    if "YOUR_USERNAME" in REPO_URL or "YOUR_REPO" in REPO_URL:
        raise ValueError("Edit REPO_URL in the settings cell before cloning.")
    clone_cmd = ["git", "clone"]
    if BRANCH:
        clone_cmd += ["--branch", BRANCH]
    clone_cmd += [REPO_URL, str(PROJECT_DIR)]
    subprocess.run(clone_cmd, check=True)
else:
    print(f"Project directory already exists: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("cwd:", Path.cwd())

requirements = PROJECT_DIR / "requirements.txt"
if requirements.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)

import torch
print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())


## 3. Locate Kaggle Input Dataset

In [ ]:
def is_public_dir(path: Path) -> bool:
    return (
        (path / "annotations" / "train.json").exists()
        and (path / "annotations" / "val.json").exists()
        and (path / "train" / "images").exists()
        and (path / "val" / "images").exists()
    )

def find_public_dir() -> Path:
    if DATASET_DIR is not None:
        candidate = Path(DATASET_DIR)
        if is_public_dir(candidate):
            return candidate
        if is_public_dir(candidate / "public"):
            return candidate / "public"
        raise FileNotFoundError(f"DATASET_DIR does not look like public dataset: {candidate}")

    input_root = Path("/kaggle/input")
    candidates = []
    for train_json in input_root.rglob("train.json"):
        if train_json.parent.name != "annotations":
            continue
        candidate = train_json.parent.parent
        if is_public_dir(candidate):
            candidates.append(candidate)
    if not candidates:
        raise FileNotFoundError("Could not find public dataset under /kaggle/input.")
    candidates = sorted(set(candidates), key=lambda item: str(item))
    print("Candidates:")
    for item in candidates:
        print(" -", item)
    return candidates[0]

PUBLIC_DIR = find_public_dir()
TRAIN_JSON = PUBLIC_DIR / "annotations" / "train.json"
VAL_JSON = PUBLIC_DIR / "annotations" / "val.json"
TRAIN_IMAGES = PUBLIC_DIR / "train" / "images"
VAL_IMAGES = PUBLIC_DIR / "val" / "images"
EVALUATOR = PUBLIC_DIR / "tools" / "evaluate_predictions.py"

print("PUBLIC_DIR:", PUBLIC_DIR)
print("TRAIN_JSON:", TRAIN_JSON)
print("VAL_JSON:", VAL_JSON)
print("TRAIN_IMAGES:", TRAIN_IMAGES)
print("VAL_IMAGES:", VAL_IMAGES)
print("EVALUATOR:", EVALUATOR, EVALUATOR.exists())


## 4. Sanity Check Dataset And Code

In [ ]:
import json

for split_name, json_path in [("train", TRAIN_JSON), ("val", VAL_JSON)]:
    data = json.loads(json_path.read_text(encoding="utf-8"))
    print(split_name, "images=", len(data["images"]), "annotations=", len(data["annotations"]))

subprocess.run([sys.executable, "-m", "py_compile", "train.py", "predict.py"], check=True)
subprocess.run([
    sys.executable, "-m", "utils.check_dataset",
    "--annotations", str(TRAIN_JSON),
    "--image_dir", str(TRAIN_IMAGES),
    "--image_size", str(IMAGE_SIZE),
    "--batch_size", "2",
    "--train",
], check=True)


## 5. Train

In [ ]:
from IPython.display import display
import pandas as pd

train_cmd = [
    sys.executable, "train.py",
    "--train_data", str(TRAIN_JSON),
    "--val_data", str(VAL_JSON),
    "--image_dir", str(TRAIN_IMAGES),
    "--val_image_dir", str(VAL_IMAGES),
    "--checkpoint_dir", str(PROJECT_DIR / "models"),
    "--image_size", str(IMAGE_SIZE),
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
    "--lr", str(LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--conf_threshold", str(CONF_THRESHOLD),
    "--nms_threshold", str(NMS_THRESHOLD),
]
if USE_AMP:
    train_cmd.append("--amp")
if USE_PRETRAINED_BACKBONE:
    train_cmd.append("--pretrained_backbone")
if FREEZE_BACKBONE_STEM:
    train_cmd.append("--freeze_backbone_stem")
if MAX_TRAIN_BATCHES > 0:
    train_cmd += ["--max_train_batches", str(MAX_TRAIN_BATCHES)]
if MAX_VAL_BATCHES > 0:
    train_cmd += ["--max_val_batches", str(MAX_VAL_BATCHES)]

print("Train command:")
print(" ".join(train_cmd))
subprocess.run(train_cmd, check=True)

history_files = sorted((PROJECT_DIR / "models").glob("train_history_*.csv"), key=lambda item: item.stat().st_mtime)
if history_files:
    HISTORY_CSV = history_files[-1]
    history_df = pd.read_csv(HISTORY_CSV)
    print("History CSV:", HISTORY_CSV)
    display_columns = [
        "epoch", "lr", "train_loss", "val_loss", "val_map50", "best_map50",
        "train_box_loss", "train_objectness_loss", "train_class_loss",
        "val_box_loss", "val_objectness_loss", "val_class_loss",
    ]
    display(history_df[[col for col in display_columns if col in history_df.columns]].tail(10))
else:
    HISTORY_CSV = None
    print("No train history CSV found.")


## 6. Predict On Validation And Evaluate

In [ ]:
VAL_PREDICTIONS = PROJECT_DIR / "val_predictions.json"
predict_val_cmd = [
    sys.executable, "predict.py",
    "--image_dir", str(VAL_IMAGES),
    "--output", str(VAL_PREDICTIONS),
    "--checkpoint", str(PROJECT_DIR / "models" / "best.pth"),
    "--batch_size", str(BATCH_SIZE),
    "--conf_threshold", str(CONF_THRESHOLD),
    "--nms_threshold", str(NMS_THRESHOLD),
]
print("Predict validation command:")
print(" ".join(predict_val_cmd))
subprocess.run(predict_val_cmd, check=True)

val_predictions_data = json.loads(VAL_PREDICTIONS.read_text(encoding="utf-8"))
num_pred_boxes = sum(len(item["boxes"]) for item in val_predictions_data)
print(f"Validation predictions: images={len(val_predictions_data)} boxes={num_pred_boxes}")
print("Prediction preview:")
print(json.dumps(val_predictions_data[:3], ensure_ascii=False, indent=2))

if EVALUATOR.exists():
    VAL_SCORE = PROJECT_DIR / "val_score.json"
    subprocess.run([
        sys.executable, str(EVALUATOR),
        "--ground_truth", str(VAL_JSON),
        "--predictions", str(VAL_PREDICTIONS),
        "--output", str(VAL_SCORE),
    ], check=True)
    score = json.loads(VAL_SCORE.read_text(encoding="utf-8"))
    print("Validation score:")
    print(json.dumps(score, ensure_ascii=False, indent=2))
    if "per_class" in score:
        per_class_df = pd.DataFrame.from_dict(score["per_class"], orient="index")
        display(per_class_df)
else:
    VAL_SCORE = None
    print("Evaluator not found, skipped local validation scoring.")


## 7. Predict On Hidden/Test Images If Available

In [ ]:
if TEST_IMAGE_DIR is None:
    print("TEST_IMAGE_DIR is None. Skipped final prediction.")
else:
    test_dir = Path(TEST_IMAGE_DIR)
    final_cmd = [
        sys.executable, "predict.py",
        "--image_dir", str(test_dir),
        "--output", str(FINAL_PREDICTIONS),
        "--checkpoint", str(PROJECT_DIR / "models" / "best.pth"),
        "--batch_size", str(BATCH_SIZE),
        "--conf_threshold", str(CONF_THRESHOLD),
        "--nms_threshold", str(NMS_THRESHOLD),
    ]
    print("Final predict command:")
    print(" ".join(final_cmd))
    subprocess.run(final_cmd, check=True)
    final_data = json.loads(FINAL_PREDICTIONS.read_text(encoding="utf-8"))
    print(f"Final predictions: images={len(final_data)} boxes={sum(len(item['boxes']) for item in final_data)}")
    print("Final prediction preview:")
    print(json.dumps(final_data[:3], ensure_ascii=False, indent=2))
    print("Final predictions path:", FINAL_PREDICTIONS)


## 8. Collect Artifacts

In [ ]:
import shutil

ARTIFACT_DIR = Path("/kaggle/working/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    PROJECT_DIR / "models" / "best.pth",
    PROJECT_DIR / "models" / "last.pth",
    VAL_PREDICTIONS,
    PROJECT_DIR / "val_score.json",
]
files_to_copy += sorted((PROJECT_DIR / "models").glob("train_history_*.csv"))
if FINAL_PREDICTIONS.exists():
    files_to_copy.append(FINAL_PREDICTIONS)

copied = []
for src in files_to_copy:
    if src.exists():
        dst = ARTIFACT_DIR / src.name
        shutil.copy2(src, dst)
        copied.append(dst)

zip_path = shutil.make_archive("/kaggle/working/object_detection_artifacts", "zip", ARTIFACT_DIR)
print("Artifacts directory:", ARTIFACT_DIR)
print("Artifacts zip:", zip_path)
print("Files:")
artifact_rows = []
for path in sorted(ARTIFACT_DIR.iterdir()):
    artifact_rows.append({"file": path.name, "size_mb": round(path.stat().st_size / (1024 * 1024), 3)})
display(pd.DataFrame(artifact_rows))


## 9. Summary Shown In Notebook

In [ ]:
summary = {
    "best_checkpoint": str(PROJECT_DIR / "models" / "best.pth"),
    "history_csv": str(HISTORY_CSV) if HISTORY_CSV is not None else None,
    "val_predictions": str(VAL_PREDICTIONS),
    "val_score": str(PROJECT_DIR / "val_score.json"),
    "artifacts_zip": "/kaggle/working/object_detection_artifacts.zip",
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

if HISTORY_CSV is not None:
    history_df = pd.read_csv(HISTORY_CSV)
    print("Final history row:")
    display(history_df.tail(1))

score_path = PROJECT_DIR / "val_score.json"
if score_path.exists():
    score = json.loads(score_path.read_text(encoding="utf-8"))
    print(f"mAP@0.5={score.get('mAP@0.5')} performance_points={score.get('performance_points')}")
